# Demo 3 — FA (Foundation Agents Protocol)

Source: [github.com/FoundationAgents/ai-link-net](https://github.com/FoundationAgents/ai-link-net)

Same scenario as Demo 2 (alice asks for a stock brief → analyst delegates to a sector specialist), but the cross-agent layer is **FA** instead of A2A. Equity **discovers** macro at runtime — no URL hardcoded anywhere.

![](../figures/fa_topology.png)

Three independent hosts under one relay; one entity per host. **Two protocols stack:** FA routes between entities (cross-host); MCP supplies each agent's tools (stock data).

## First-time setup

Run this once in a terminal, from the directory where you want the repo:

```bash
git clone https://github.com/jackwu502/ivado-protocol.git
cd ivado-protocol

python3.12 -m venv .venv
source .venv/bin/activate

python -m pip install --upgrade pip
python -m pip install -r requirements.txt
python -m ipykernel install --user --name ivado-lab --display-name "ivado-lab (3.12)"
```

Create your local `.env` file:

```bash
cp .env.example .env
```

Then edit `.env` and fill in one credential route:

```bash
# Option 1: Anthropic direct
ANTHROPIC_API_KEY=sk-ant-...
ANTHROPIC_MODEL=claude-sonnet-4-6

# Option 2: OpenRouter-compatible Anthropic endpoint
# ANTHROPIC_BASE_URL=https://openrouter.ai/api
# ANTHROPIC_API_KEY=sk-or-v1-...
# ANTHROPIC_MODEL=anthropic/claude-sonnet-4.5
```

Do not commit `.env`; it is intentionally gitignored.

Start Jupyter from the repo root and select the `ivado-lab (3.12)` kernel:

```bash
python -m jupyter lab
```


Needs the FA reference implementation [`ai-link-net`](https://github.com/FoundationAgents/ai-link-net). The install cell below uses a local clone if present, otherwise GitHub.

In [1]:
# ── Bootstrap: resolve paths to shared/ and sibling helpers ──
import sys
from pathlib import Path
HERE = Path.cwd()
ROOT = HERE.parent
sys.path.insert(0, str(ROOT))   # so `from shared.X import Y` works
sys.path.insert(0, str(HERE))   # so sibling helpers import directly
STOCK_MCP_SERVER = str(ROOT / "shared" / "stock_mcp_server.py")


In [2]:
assert sys.version_info >= (3, 12), "ai-link-net needs Python 3.12+"

# Dependencies for the wrapped MCP server and the analyst loop
%pip install -q mcp anthropic yfinance python-dotenv

import importlib.util
import subprocess

def _have_ai_link_net():
    return importlib.util.find_spec("fp") is not None and importlib.util.find_spec("aln") is not None

def _pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])

if _have_ai_link_net():
    print("ai-link-net already importable")
else:
    candidates = []
    candidates.append(ROOT / "ai-link-net")
    candidates.append(ROOT.parent / "ai-link-net")

    local_clone = next((p for p in candidates if p and p.exists()), None)
    if local_clone:
        print(f"installing ai-link-net from local clone: {local_clone}")
        _pip_install("-e", str(local_clone))
    else:
        print("installing ai-link-net from GitHub")
        _pip_install('git+https://github.com/FoundationAgents/ai-link-net.git')

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
print("Python:", sys.version.split()[0])

Note: you may need to restart the kernel to use updated packages.
ai-link-net already importable
Python: 3.12.4


## Setup

Imports plus three small patches for a clean in-notebook demo: silence FA's chatty info-level logs, skip its on-disk persistence, and teach `Host` to resolve `TOOL` entities through `MCPHandler` (the bridge from FA's tool concept to MCP).

In [3]:
import asyncio, json
from unittest.mock import patch
from loguru import logger

logger.remove()
logger.add(sys.stderr, level="WARNING", format="<level>{level}</level> | {message}")
patch("fp.host.Host.save").start()

from fp import Host
from aln.app.handlers import create_entity_handler

_orig = Host._resolve_entity_handler
def _resolve(self, entity, handler, provider, system_prompt, handler_config):
    return _orig(self, entity, handler, provider, system_prompt, handler_config) \
        or create_entity_handler(entity=entity, kind=entity.kind, provider=provider,
                                 system_prompt=system_prompt, handler_config=handler_config)
Host._resolve_entity_handler = _resolve
print("ready")

ready


## 1. System prompts + tool schemas

Two system prompts (one per agent) plus two extra tools that equity will use to discover and delegate: `list_network_specialists` and `delegate_to_specialist`.

Notice the tool definitions are **completely generic** — no mention of "macro" anywhere. The equity prompt tells Claude to *first* list what's on the network, *read each entity's description*, and pick whichever one fits the question. That's the FA discovery story in code form.

In [4]:
from fp import Host, Message, MessageKind
from fp.core.base import EntityKind
from fp.core.wellknown import FPAddress
from fp.message import FriendRequestPayload
from shared.agent_runner import run_agent

FRIEND_KINDS = {MessageKind.FRIEND_REQUEST, MessageKind.FRIEND_ACCEPT, MessageKind.FRIEND_REJECT}

MACRO_SYSTEM_PROMPT = (
    "You are a macro / sector analyst. Given a sector or thematic question, "
    "answer concisely (2-3 sentences) using the available stock-data tools "
    "if helpful. Do not produce long reports."
)

EQUITY_SYSTEM_PROMPT = (
    "You are an equity analyst writing a brief on a single stock. Use the "
    "stock-data tools to fetch price action, company info, and news.\n\n"
    "If sector or macro context would inform your brief, you may delegate "
    "that question to another agent on the federated network. The procedure:\n"
    "  1) Call `list_network_specialists` to see what other public agents "
    "     are reachable. Each entry has an entity_id and a description.\n"
    "  2) Read the descriptions and pick the one whose stated expertise "
    "     best matches what you need.\n"
    "  3) Call `delegate_to_specialist(entity_id=..., question=...)` with "
    "     the chosen entity_id.\n\n"
    "Use at most one delegation per brief. Then write a concise brief that "
    "weaves in the delegated answer if you obtained one."
)

LIST_SPECIALISTS_TOOL = {
    "name": "list_network_specialists",
    "description": (
        "List public AGENT entities reachable on the federated network "
        "(excluding yourself). Returns a JSON array of objects with "
        "entity_id, name, and description."
    ),
    "input_schema": {"type": "object", "properties": {}},
}

DELEGATE_TO_SPECIALIST_TOOL = {
    "name": "delegate_to_specialist",
    "description": (
        "Send a plain-English question to another agent on the network "
        "and return its answer. You must first call "
        "list_network_specialists to obtain a valid entity_id."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "entity_id": {"type": "string"},
            "question": {"type": "string"},
        },
        "required": ["entity_id", "question"],
    },
}

def _payload_text(msg):
    p = msg.payload
    if isinstance(p, dict):
        return p.get("text") or p.get("question") or ""
    return getattr(p, "text", "") or ""

def _is_agent(card):
    return "agent" in str(card.kind).lower()

## 2. Macro analyst

A simple FA entity: when it receives an INVOKE mail, it runs a Claude loop with the stock MCP server and sends the answer back as another mail. No A2A involved — the cross-agent layer here is FA itself.

In [5]:
def make_macro_analyst(host, name="MacroAnalyst"):
    state = {}

    async def _handle_query(sender_addr, msg):
        question = _payload_text(msg)
        try:
            answer = await run_agent(
                question=question,
                mcp_servers=[STOCK_MCP_SERVER],
                system_prompt=MACRO_SYSTEM_PROMPT,
            )
        except Exception as exc:
            answer = f"(macro analyst error: {exc})"
        await state["entity"].send_message(
            to=FPAddress(address=sender_addr),
            message=Message(kind=MessageKind.INVOKE, payload={"text": answer}),
        )

    async def handler(msg):
        if msg.kind in FRIEND_KINDS:
            return
        sender_addr = msg.metadata.get("sender_address", "")
        if not sender_addr:
            return
        asyncio.create_task(_handle_query(sender_addr, msg))

    entity = host.register_entity(
        name=name,
        kind=EntityKind.AGENT,
        is_public=True,
        description="Sector / macro outlook analyst.",
        handler=handler,
    )
    state["entity"] = entity
    return entity

## 3. Equity analyst (discovery + delegation)

Three pieces to look at:

- **`_list_specialists`** — calls `host.get_discoverable_entities(include_parent=True)`, returns the list with descriptions, and prints each one so you can watch the discovery happen.
- **`_ask(entity_id, question)`** — sends an INVOKE to whichever entity Claude picked, then awaits the reply on a queue.
- **`handler`** — when something arrives, route it: a reply from a specialist we're waiting on goes into the queue; anything else is treated as a new user query.

Crucially, equity's source has **zero hardcoded knowledge** about macro — no name, no URL, no entity_id.

In [6]:
def make_equity_analyst(host, name="EquityAnalyst"):
    state = {"entity": None, "pending_target_id": None, "pending_queue": None}

    def _list_specialists():
        me_id = state["entity"].address.entity_uid
        items = []
        for card in host.get_discoverable_entities(include_parent=True):
            if not _is_agent(card): continue
            if card.address.entity_uid == me_id: continue
            items.append({
                "entity_id": card.address.address,
                "name": card.name,
                "description": (card.description or "").strip(),
            })
        # Visible trace
        print(f"  [equity → discovery] list_network_specialists "
              f"returned {len(items)} agents:")
        for it in items:
            desc = it["description"][:60]
            print(f"    • {it['name']:<14}  ({it['entity_id']})  — {desc}")
        return json.dumps(items, indent=2)

    def _find_card_by_id(entity_id):
        for card in host.get_discoverable_entities(include_parent=True):
            if card.address.address == entity_id:
                return card
        return None

    async def _ask(entity_id, question):
        target = _find_card_by_id(entity_id)
        if target is None:
            return f"(no entity with id {entity_id} on the network)"
        print(f"  [equity → delegation] delegate_to_specialist "
              f"to {target.name}: \"{question[:80]}\"")
        # Lazy friend handshake
        if target.address.entity_uid not in state["entity"].friends:
            await state["entity"].send_message(
                to=target,
                message=Message(
                    kind=MessageKind.FRIEND_REQUEST,
                    payload=FriendRequestPayload(sender_card=state["entity"].entity_card),
                ),
            )
            await asyncio.sleep(0.5)
        # Route INVOKE, wait for reply via inbox queue
        queue = asyncio.Queue()
        state["pending_target_id"] = target.address.address
        state["pending_queue"] = queue
        await state["entity"].send_message(
            to=target,
            message=Message(kind=MessageKind.INVOKE, payload={"text": question}),
        )
        try:
            reply = await asyncio.wait_for(queue.get(), timeout=120)
            text = _payload_text(reply) or "(empty reply)"
            print(f"  [equity ← reply] from {target.name}: {len(text)} chars")
            return text
        finally:
            state["pending_target_id"] = None
            state["pending_queue"] = None

    async def _delegate(name_, args):
        if name_ == "list_network_specialists":
            return _list_specialists()
        if name_ == "delegate_to_specialist":
            return await _ask(args.get("entity_id", ""), args.get("question", ""))
        return f"(unknown tool: {name_})"

    async def _handle_query(sender_addr, msg):
        question = _payload_text(msg)
        try:
            answer = await run_agent(
                question=question,
                mcp_servers=[STOCK_MCP_SERVER],
                system_prompt=EQUITY_SYSTEM_PROMPT,
                extra_tools=[LIST_SPECIALISTS_TOOL, DELEGATE_TO_SPECIALIST_TOOL],
                extra_tool_executor=_delegate,
            )
        except Exception as exc:
            answer = f"(equity analyst error: {exc})"
        await state["entity"].send_message(
            to=FPAddress(address=sender_addr),
            message=Message(kind=MessageKind.INVOKE, payload={"text": answer}),
        )

    async def handler(msg):
        if msg.kind in FRIEND_KINDS: return
        sender_addr = msg.metadata.get("sender_address", "")
        if not sender_addr: return
        # Reply from a specialist we are currently waiting on → queue
        if (state["pending_target_id"] == sender_addr
                and state["pending_queue"] is not None):
            await state["pending_queue"].put(msg)
            return
        # Anything else: new user query
        asyncio.create_task(_handle_query(sender_addr, msg))

    entity = host.register_entity(
        name=name,
        kind=EntityKind.AGENT,
        is_public=True,
        description=(
            "Equity analyst — writes briefs on individual stocks. May "
            "delegate sector/macro questions to other agents on the network."
        ),
        handler=handler,
    )
    state["entity"] = entity
    return entity

## 4. Build the network

Four hosts under one relay: `CloudHost` (root), `AliceHost` (alice), `EquityHost` (equity), `MacroHost` (macro). Each host has exactly one entity — they're truly independent.

**Alice has no idea macro exists** — she only knows her own host. Equity will discover macro at query time through the federation tree.

In [7]:
cloud      = Host(name="CloudHost")
host_alice = Host(name="AliceHost",  port=18100)
host_equity = Host(name="EquityHost", port=18101)
host_macro  = Host(name="MacroHost",  port=18102)
for h in (host_alice, host_equity, host_macro):
    h.set_parent_host(cloud)

# Each agent on its own host
macro  = make_macro_analyst(host_macro)
equity = make_equity_analyst(host_equity)

# Alice (the user) on her own host — separate from any agent
alice_inbox: asyncio.Queue[Message] = asyncio.Queue()
async def alice_handler(msg):
    if msg.kind not in FRIEND_KINDS:
        await alice_inbox.put(msg)
alice = host_alice.register_entity("alice", kind=EntityKind.HUMAN, handler=alice_handler)

print(f"{'Entity':<14}  {'Address':<70}  Lives on")
for ent, hname in [(alice, 'AliceHost'), (equity, 'EquityHost'), (macro, 'MacroHost')]:
    print(f"{ent.name:<14}  {ent.address.address:<70}  {hname}")

Entity          Address                                                                 Lives on
alice           b1bb75b8:c9f964a7                                                       AliceHost
EquityAnalyst   718bab64:e28501df                                                       EquityHost
MacroAnalyst    06df5474:07255930                                                       MacroHost


## 5. Discovery (the FA-only primitive)

Easiest place to inspect the federation is **on the relay** (`cloud`) — it sees all its children directly. **No URL was configured anywhere**; the host network IS the directory.

You'll only see **public** entities. Alice is registered without `is_public=True` (default is private), so she doesn't appear — humans typically aren't broadcast as callable services. Only the two AGENT entities show up, which is exactly what equity will pick from at runtime.

> **At runtime equity doesn't have access to `cloud`** — it only knows its own host. So it calls `host_equity.get_discoverable_entities(include_parent=True)`, which walks up through `parent_host` to cloud and gets the same list back. Same result, different POV.

In [8]:
print("From the relay's POV — all public entities reachable in the federation:\n")
for card in cloud.get_discoverable_entities(include_parent=False):
    print(f"  {card.name:<14}  {card.address.address}  (kind={card.kind})")
    print(f"    description: {card.description}")
print("\n(alice is hidden — not is_public)")

From the relay's POV — all public entities reachable in the federation:

  EquityAnalyst   718bab64:e28501df  (kind=agent)
    description: Equity analyst — writes briefs on individual stocks. May delegate sector/macro questions to other agents on the network.
  MacroAnalyst    06df5474:07255930  (kind=agent)
    description: Sector / macro outlook analyst.

(alice is hidden — not is_public)


## 6. Friend handshake (alice ↔ equity)

FA requires a one-time friend handshake before two entities exchange real messages — it's how each side captures the other's signing key. Alice ↔ equity we do explicitly here; the equity ↔ macro handshake will happen lazily inside `_ask` the first time equity delegates.

In [9]:
await alice.send_message(
    to=equity.entity_card,
    message=Message(kind=MessageKind.FRIEND_REQUEST,
                    payload=FriendRequestPayload(sender_card=alice.entity_card)),
)
await asyncio.sleep(0.5)
print("alice friends:", list(alice.friends.keys()))

alice friends: ['e28501df']


## 7. Alice asks one question

This is the moment of truth. Alice sends one INVOKE to equity. Watch the printed trace:

- `[equity → discovery]` — equity asks the network "who's reachable?" and the host returns a list of EntityCards
- equity's Claude reads each `description` and **picks** the right specialist (it's never told the name "macro")
- `[equity → delegation]` — equity sends the question, routed `EquityHost → CloudHost → MacroHost`
- macro's reply travels back the same path
- equity weaves it all into a final brief

In [10]:
await alice.send_message(
    to=equity.entity_card,
    message=Message(
        kind=MessageKind.INVOKE,
        payload={"text": "Analyze NVDA briefly. Include sector context."},
    ),
)
reply = await asyncio.wait_for(alice_inbox.get(), timeout=300)
print(reply.payload["text"])

  [equity → discovery] list_network_specialists returned 1 agents:
    • MacroAnalyst    (06df5474:07255930)  — Sector / macro outlook analyst.


  [equity → delegation] delegate_to_specialist to MacroAnalyst: "What is the current outlook for the semiconductor sector, particularly AI chip m"


  [equity ← reply] from MacroAnalyst: 672 chars


---

## NVIDIA Corporation (NVDA) - Equity Brief

**Price Action:** NVDA closed at $198.45 on May 1, 2026, down 0.56% (-$1.12) on the day with elevated volume of 128M shares. The stock has shown strong momentum over the past 30 days, rallying from $172.70 on March 20 to a peak of $216.61 on April 27 (+25%), before pulling back ~8% in the final three trading days of the month. This recent volatility suggests profit-taking after a strong run.

**Company Profile:** NVIDIA ($4.82T market cap) is the dominant player in AI accelerated computing and data center infrastructure within the semiconductor industry. The company operates through Compute & Networking (data center AI solutions) and Graphics (GeForce gaming GPUs) segments.

**Sector Context:** The semiconductor AI chip sector is experiencing mixed momentum but remains fundamentally strong. Massive AI data center buildouts by hyperscalers (Alphabet, Meta, Microsoft) continue to drive demand, with Asian chipmakers also reaching record le

### Recap

Look for the "Sector Context" paragraph in the brief above — that text was written by macro on MacroHost and travelled `MacroHost → CloudHost → EquityHost`, end-to-end. And the discovery line proves equity picked macro purely from the description string — re-run with a "geopolitics analyst" instead and it would just work, no code change.

Three FA primitives all fired in sequence: **discovery** (`get_discoverable_entities`), **Entity-ID addressing** (`delegate_to_specialist` takes an entity_id, not a URL), and **cross-host routing** (cloud relayed without either child knowing the other's IP). In A2A all three would be the application's problem; in FA they're protocol-native.

## 8. Bonus — offline delivery (something A2A cannot do)

**The story:** alice2 keeps sending messages to a worker. Halfway through, the worker's machine drops off the network. In **A2A** the call would just fail. In **FA** the relay quietly **queues** the messages and delivers them when the worker comes back — alice2 never knows anything went wrong.

We'll run a tiny 3-host federation (real WebSockets, real disconnect) and watch this happen.

> **macOS first-run only:** alias the extra loopbacks once (sticks until reboot):
> ```bash
> sudo ifconfig lo0 alias 127.0.0.2 up
> sudo ifconfig lo0 alias 127.0.0.3 up
> ```

### Setup

Plumbing — `HostServer` imports + a small helper to start each host on its own loopback IP. Skim it; the interesting code is in the next cells.

In [11]:
import uvicorn
from fastapi import FastAPI
from fp import EntityStatus
from aln.app.service.host_server import HostServer
from aln.app.api.ws import router as ws_router
from aln.app.api.well_known import router as well_known_router

# Skip disk persistence for the in-notebook demo
patch("aln.app.service.host_server.HostServer._save_offline_mail_queues").start()
patch("aln.app.service.host_server.HostServer._load_offline_mail_queues").start()

def _make_host_app(host_runtime: HostServer) -> FastAPI:
    app = FastAPI()
    app.include_router(well_known_router)
    app.include_router(ws_router)
    app.state.host_runtime = host_runtime
    return app

async def start_host(host_runtime: HostServer, host: str, port: int):
    config = uvicorn.Config(_make_host_app(host_runtime), host=host, port=port, log_level="warning")
    server = uvicorn.Server(config)
    server.install_signal_handlers = lambda: None
    asyncio.create_task(server.serve())
    for _ in range(50):
        await asyncio.sleep(0.1)
        if server.started: break
    return server

async def _no_reconnect():  # disable auto-retry so we control online/offline manually
    pass

print("helpers ready")

helpers ready


### Build the federation

Three machines:

| Where | What lives there |
|---|---|
| **127.0.0.1**  (relay)  | nothing — just routes mail |
| **127.0.0.2**  (host_x) | `alice2` (the user) |
| **127.0.0.3**  (host_y) | `worker` (an echo agent) |

host_x and host_y open outbound WebSockets to the relay. That handshake is what marks them ONLINE.

In [12]:
# Build the three hosts
relay  = HostServer(name="Relay",  port=20001)
host_x = HostServer(name="HostX",  port=20002)  # alice2 lives here
host_y = HostServer(name="HostY",  port=20003)  # worker lives here

# alice2 — collects replies in an inbox queue
inbox2: asyncio.Queue = asyncio.Queue()
async def alice2_handler(msg):
    if msg.kind not in FRIEND_KINDS:
        await inbox2.put(msg)
alice2 = host_x.register_entity("alice2", kind=EntityKind.HUMAN, handler=alice2_handler)

# worker — echoes whatever it receives
async def worker_handler(msg):
    if msg.kind in FRIEND_KINDS: return
    sender = msg.metadata.get("sender_address", "")
    text = msg.payload.get("text", "")
    await worker.send_message(
        to=FPAddress(address=sender),
        message=Message(kind=MessageKind.INVOKE, payload={"text": f"echo: {text}"}),
    )
worker = host_y.register_entity(
    "worker", kind=EntityKind.AGENT, is_public=True,
    description="echo worker", handler=worker_handler,
)
host_y._reconnect_to_parent = _no_reconnect  # we control reconnects manually

# Boot uvicorn for each host on its own loopback IP
relay_srv  = await start_host(relay,  "127.0.0.1", 20001)
host_x_srv = await start_host(host_x, "127.0.0.2", 20002)
host_y_srv = await start_host(host_y, "127.0.0.3", 20003)

# Children connect outbound to the relay
await host_x.connect_to_parent("http://127.0.0.1:20001")
await host_y.connect_to_parent("http://127.0.0.1:20001")
await asyncio.sleep(1.0)


# Helper: print a clean snapshot of "what relay sees" + alice2's inbox
def show_state(label: str):
    worker_status = relay.entity_status.get(worker.address.entity_uid)
    queued = sum(len(q) for q in relay.offline_mail_queues.values())
    inbox = inbox2.qsize()
    status_str = worker_status.value.upper() if worker_status else "?"
    print(f"\n─── State: {label} ───")
    print(f"   worker (relay's view):    {status_str}")
    print(f"   messages queued at relay: {queued}")
    print(f"   alice2's inbox:           {inbox} reply/replies waiting")
    print(f"───────────────────────────────────────────")

show_state("after everyone connects")


─── State: after everyone connects ───
   worker (relay's view):    ONLINE
   messages queued at relay: 0
   alice2's inbox:           0 reply/replies waiting
───────────────────────────────────────────


### Baseline — round-trip while everyone is online

One quick exchange to confirm the network actually routes.

In [13]:
# Friend handshake (silent — required before sending real mail)
await alice2.send_message(
    to=worker.entity_card,
    message=Message(kind=MessageKind.FRIEND_REQUEST,
                    payload=FriendRequestPayload(sender_card=alice2.entity_card)),
)
await asyncio.sleep(0.8)

# Send and receive
await alice2.send_message(
    to=worker.entity_card,
    message=Message(kind=MessageKind.INVOKE, payload={"text": "baseline"}),
)
reply = await asyncio.wait_for(inbox2.get(), timeout=5.0)
print(f"alice2 sent 'baseline' → got back '{reply.payload['text']}'")
show_state("after baseline round-trip")

alice2 sent 'baseline' → got back 'echo: baseline'

─── State: after baseline round-trip ───
   worker (relay's view):    ONLINE
   messages queued at relay: 0
   alice2's inbox:           0 reply/replies waiting
───────────────────────────────────────────


### Worker drops off the network

We close host_y's WebSocket — same effect as worker's machine being shut down or losing wifi. **The relay notices on its own** (no manual flag flipping) and marks worker OFFLINE.

In [14]:
print("[host_y disconnects from the relay…]")
await host_y.disconnect_from_parent()
await asyncio.sleep(1.5)  # let the relay notice the dropped connection

show_state("after worker goes offline")

[host_y disconnects from the relay…]



─── State: after worker goes offline ───
   worker (relay's view):    OFFLINE
   messages queued at relay: 0
   alice2's inbox:           0 reply/replies waiting
───────────────────────────────────────────


### alice2 sends 3 messages anyway

alice2's code didn't change — she has no idea worker is down. The relay catches the OFFLINE status and **queues each message** instead of failing. Watch the queue grow.

In [15]:
print("alice2 sends 3 messages to the (offline) worker…")
for i in range(3):
    await alice2.send_message(
        to=worker.entity_card,
        message=Message(kind=MessageKind.INVOKE, payload={"text": f"queued-{i+1}"}),
    )
await asyncio.sleep(1.0)

show_state("3 messages later (worker still offline)")

alice2 sends 3 messages to the (offline) worker…



─── State: 3 messages later (worker still offline) ───
   worker (relay's view):    OFFLINE
   messages queued at relay: 3
   alice2's inbox:           0 reply/replies waiting
───────────────────────────────────────────


### Worker comes back

host_y reopens its WebSocket. The relay's handshake handler fires `_mark_child_entities_online` AND `_flush_offline_queues_for_child` automatically — the 3 queued messages stream out, worker echoes them, replies route back to alice2.

In [16]:
print("[host_y reconnects to the relay…]")
await host_y.connect_to_parent("http://127.0.0.1:20001")
await asyncio.sleep(2.0)  # handshake + auto-flush + worker replies + routing

show_state("after worker reconnects (relay auto-flushed)")

print("\nalice2 finally received:")
while not inbox2.empty():
    reply = inbox2.get_nowait()
    print(f"   ← {reply.payload['text']}")

[host_y reconnects to the relay…]



─── State: after worker reconnects (relay auto-flushed) ───
   worker (relay's view):    ONLINE
   messages queued at relay: 0
   alice2's inbox:           3 reply/replies waiting
───────────────────────────────────────────

alice2 finally received:
   ← echo: queued-1
   ← echo: queued-2
   ← echo: queued-3


### Recap

Look back at the four `─── State ───` blocks:

1. Everyone online → worker ONLINE, queue 0
2. Worker disconnects → worker OFFLINE, queue 0
3. alice2 sends 3 → worker OFFLINE, **queue 3**
4. Worker reconnects → worker ONLINE, **queue 0** + 3 replies in alice2's inbox

alice2's code is identical the whole time. **No retry loop. No "is recipient online?" check. No timeout handling.** The relay handled everything because addressing is by stable Entity ID and queueing is a protocol primitive.

(In production the relay also writes the queue to disk — survives restarts. Turned off here for the in-notebook demo.)

### Cleanup

In [17]:
for srv in (relay_srv, host_x_srv, host_y_srv):
    srv.should_exit = True
await asyncio.sleep(0.5)
print("offline-demo subnet stopped")

offline-demo subnet stopped


## CLI entry point

The non-coding workflow uses the `aln` CLI and a WebUI (`aln init`, `aln ui`, `bash demo/quickstart.sh`). Below is just the help text.

In [18]:
import shutil, subprocess
from pathlib import Path
aln = shutil.which("aln") or str(Path(sys.executable).parent / "aln")
if Path(aln).exists():
    print(subprocess.run([aln, "-h"], capture_output=True, text=True).stdout[:1200])
else:
    print("`aln` not on PATH (does not affect the demo above)")

Usage: fp [OPTIONS] COMMAND [ARGS]...

AI-Link-Net: A entity-to-entity communication system base on Foundation
Protocol.

→ No hosts configured yet. Start with: aln host init

These are common commands used in various situations:

messaging
  mail     Send message — the ONLY way to reply to others and report to your
           owner.
  mailbox  View conversation history and manage messages. Use this to recall
           context when unsure what was discussed.

social & discovery
  find    Discover entities across the network from an entity's host
          perspective.
  friend  Friend management - entity-initiated social connections.

trade & payment
  market    Market — publish, discover, and negotiate orders across
            categories.
  pay       Payment operations — collect, confirm, balance.
  contract  Contract lifecycle management.

System commands are hidden — only the owner can run them.
Use `aln -h --full` to see all commands.

Global options:

  --host TEXT         Host 